## ***IMPORT LIBRERIE***


In [1]:
import os
import pandas as pd
import numpy as np
from app_config import REPORTS_DIR, FIGURES_DIR, MODELS_DIR
import sliding_window_on_data
from torch.utils.data import DataLoader
import glob

from models.DeepConvLSTM import DeepConvLSTM, HARDataset 
import optuna
from optuna.pruners import MedianPruner
import optuna.visualization as vis
import train
import torch
import torch.nn as nn
import train_with_cm
import matplotlib.pyplot as plt
#definisco il path da cui leggere i .csv

from utils.log_config import logger
from figures import plot_CM

#definisco il path da cui leggere i .csv

path='C:\codes\HumanActivityRecognition\data\pdd_data'
logger.debug(path)

2025-05-09 14:41:57.681 | INFO     | app_config:<module>:11 - PROJ_ROOT path is: C:\codes\HumanActivityRecognition
2025-05-09 14:42:05,189 - INFO - myapp - Logging configured from C:\codes\HumanActivityRecognition\HumanActivityRecognition\utils\base_config.json
2025-05-09 14:42:11,887 - INFO - myapp - GPU not available, training on CPU.
c:\Users\carol\Desktop\virtual_envs\har_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-09 14:42:12,687 - DEBUG - matplotlib - matplotlib data path: c:\Users\carol\Desktop\virtual_envs\har_venv\lib\site-packages\matplotlib\mpl-data
2025-05-09 14:42:12,716 - DEBUG - matplotlib - CONFIGDIR=C:\Users\carol\.matplotlib
2025-05-09 14:42:12,745 - DEBUG - matplotlib - interactive is False
2025-05-09 14:42:12,748 - DEBUG - matplotlib - platform is win32
2025-05-09 14:42:12,845 

## ***DATA PREPROCESSING***

*scansiono recording e filtro per righe non nulle*  
*OUTPUT: unico dataframe con tutte le attività non nulle*

In [2]:
#Visto che l'informazione relativa all'id del bambino lo abbiamo, così come abbiamo anche l'informazione relativa
#al giocattolo, vado a filtrare i .csv in modo tale da avere solo le righe che hanno l'attività non nulla e poi
#appendo tutte le righe delle righe non nulle in un unico dataframe
#per tutti i file che terminano in .csv nella cartella path
# Definisco il percorso della cartella contenente i CSV

# Nome del file CSV finale
final_csv_path = os.path.join(path, 'df_CAR_non_null.csv')

# Controllo se il file esiste già
if os.path.exists(final_csv_path):
    logger.debug(f"Il file {final_csv_path} esiste già. Lo sto caricando...")
    df_car = pd.read_csv(final_csv_path)
else:
    #trovo tutti i file che corrispondono a "BA" nella cartella path e li stampo a schermo
    files = glob.glob(os.path.join(path, "*_C*.csv"))
    logger.debug(f"File trovati: {files}")
    
    kid_car, kid_car_no_null = [], [] # liste per salvare utenti prima e dopo il merge 
    df_list_car = [] # lista vuota per appendere i dataframe con attività non nulla

    for file in files:
        df = pd.read_csv(file)
        
        logger.debug(f"Shape originale del file {file}: {df.shape}")
        kid_car.append(df['kid_id'].unique())
        df = df[df['action_id'] != 0] #filtro le righe con action_id non nullo
        logger.debug(f"Shape dopo il filtro del file {file}: {df.shape}")
        kid_car_no_null.append(df['kid_id'].unique())

        logger.debug(f"Colonne del file {file}: {df.columns}")
        logger.debug(f"Conteggio delle azioni nel file {file}:\n{df['action'].value_counts()}")
        logger.debug(f"Conteggio dei giocattoli nel file {file}:\n{df['toy_id'].value_counts()}")
        logger.debug("="*50)

    # mi stampo gli utenti prima di fare il merge e dopo il merge
    logger.info(f"Numero di utenti che hanno fatto almeno un'azione: {len(kid_car_no_null)/len(kid_car):.2f}")

    '''
    Visto che l'informazione relativa all'id del bambino lo abbiamo, così come abbiamo anche l'informazione relativa
    al giocattolo, vado a filtrare i .csv in modo tale da avere solo le righe che hanno l'attività non nulla e poi
    appendo tutte le righe non nulle in un unico dataframe per tutti i file che terminano in .csv nella balltella path
    '''

    for file in os.listdir(path):
        if not file.endswith('.csv'):
            continue
    
        #leggo solo i file che dopo l'undescore ha C*.csv
        if file.split('_')[-1].startswith('C') and file.endswith('.csv'): #controllo che il file termini con .csv
            df_temp=pd.read_csv(os.path.join(path,file))
            df_temp = df_temp[df_temp['action_id'] != 0]
            df_list_car.append(df_temp)

    df_car = pd.concat(df_list_car)
    logger.info(f"Dimensioni del df_car con tutte le attività non nulle: {df_car.shape}")
    logger.info(f"Colonne del df_car:\n{df_car.columns}")
    logger.info(f"Conteggio delle azioni nel df_car:\n{df_car['action'].value_counts()}")

    # salvo il dataframe
    df_car.to_csv(os.path.join(path,'df_CAR_non_null.csv'),index=False)
    logger.info(f"File salvato come {final_csv_path}")

2025-05-09 14:42:23,489 - DEBUG - myapp - Il file C:\codes\HumanActivityRecognition\data\pdd_data\df_CAR_non_null.csv esiste già. Lo sto caricando...


*per ogni attività trovata, salvo un .csv*  
*OUTPUT: un .csv per ogni attività non nulla (df_car_action_11.csv, df_car_action_19.csv ecc... )*


In [3]:
#ora divido il dataframe in base all'attività (action_id) e salvo i dataframe in un file .csv
#per ogni attività

for action_id in df_car['action_id'].unique():
    df_action = df_car[df_car["action_id"] == action_id] #filtro il dataframe in base all'attività
    logger.debug(f"Dimensioni del dataframe df_car_action_{action_id} - {df_action.shape}") #log delle dimensioni del dataframe
    logger.info(f"Conteggio delle attività per df_car_action_{action_id}") #log del conteggio delle attività
    #salvo il dataframe
    df_action.to_csv(os.path.join(path,f'df_car_action_{action_id}.csv'),index=False) #index=False per non salvare l'indice
    logger.debug(f"Salvato il dataframe df_car_action_{action_id}.csv")

2025-05-09 14:42:26,629 - DEBUG - myapp - Dimensioni del dataframe df_car_action_19 - (2617, 25)
2025-05-09 14:42:26,635 - INFO - myapp - Conteggio delle attività per df_car_action_19


2025-05-09 14:42:26,701 - DEBUG - myapp - Salvato il dataframe df_car_action_19.csv
2025-05-09 14:42:26,708 - DEBUG - myapp - Dimensioni del dataframe df_car_action_25 - (37118, 25)
2025-05-09 14:42:26,709 - INFO - myapp - Conteggio delle attività per df_car_action_25
2025-05-09 14:42:27,518 - DEBUG - myapp - Salvato il dataframe df_car_action_25.csv
2025-05-09 14:42:27,522 - DEBUG - myapp - Dimensioni del dataframe df_car_action_21 - (1925, 25)
2025-05-09 14:42:27,523 - INFO - myapp - Conteggio delle attività per df_car_action_21
2025-05-09 14:42:27,566 - DEBUG - myapp - Salvato il dataframe df_car_action_21.csv
2025-05-09 14:42:27,569 - DEBUG - myapp - Dimensioni del dataframe df_car_action_32 - (800, 25)
2025-05-09 14:42:27,570 - INFO - myapp - Conteggio delle attività per df_car_action_32
2025-05-09 14:42:27,590 - DEBUG - myapp - Salvato il dataframe df_car_action_32.csv
2025-05-09 14:42:27,594 - DEBUG - myapp - Dimensioni del dataframe df_car_action_3 - (7924, 25)
2025-05-09 14:42

## ***DATA PROCESSING***

*SLIDING WINDOW*  
*OUTPUT: unica matrice con tutte le windows concatenate*

In [4]:
#applico sliding window con la funzion process_csv
nb_sensor_channels = 9
sliding_window_length = 100
sliding_window_step = 50

#ora applico la funzione sliding window (che mi da come output x_window e y_window) a tutti i .csv relativi al giocattolo ball
#e poi concateno tutto in un unica x e y 

X, Y = [], []
kid_action_counts = {}
kid_ids = []

for action_file in [f for f in os.listdir(path) if f.endswith('.csv') and f.split('_')[1] == 'car']:
    file_path = os.path.join(path, action_file)
    action = action_file.split('_')[-1].split('.')[0]

    X_windows, Y_windows, kid_id_action_dict, kid_ids_windows = sliding_window_on_data.process_csv(file_path, nb_sensor_channels, sliding_window_length, sliding_window_step)

    X.append(X_windows)
    Y.append(Y_windows)
    kid_ids.append(kid_ids_windows)


    logger.info(f"Numero  totale di finestre per l'azione {action}:{len(X_windows)}")
    kid_action_counts[f"car_action_{action}"] = kid_id_action_dict #aggiungo il dizionario al dizionario principale per tenere traccia del numero di finestre per ogni bambino per ogni azione, ogni ball_action è una chiave e il valore è un dizionario con il numero di finestre per ogni bambino
    logger.info(f"Contenuto finale di kid_action_counts: {kid_action_counts}") #per veere quante finestre per ogni azione e per ogni bambino sono state elaborte 


# Concateno tutti i dati in un unico array per X e Y
X = np.concatenate(X, axis=0)
Y = np.concatenate(Y, axis=0)
kid_ids = np.concatenate(kid_ids, axis=0)


# Stampo le dimensioni di X e Y
logger.info(f"Dimensioni di X finale: {X.shape}")
logger.info(f"Dimensioni di Y finale: {Y.shape}")
logger.info(f"Dimensioni finali di kid_ids: {kid_ids.shape}")

2025-05-09 14:42:33,282 - DEBUG - myapp - sto leggendo il file csv: C:\codes\HumanActivityRecognition\data\pdd_data\df_car_action_10.csv
2025-05-09 14:42:33,291 - INFO - myapp - Kid_id: 3006, X_kid shape: (1950, 9), Y_kid shape: (1950,)
2025-05-09 14:42:33,294 - INFO - myapp - Numero di finestre estratte: 38
2025-05-09 14:42:33,296 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 0
2025-05-09 14:42:33,297 - INFO - myapp - Numero totale di finestre (dopo padding finale): 38
2025-05-09 14:42:33,300 - DEBUG - myapp - X_windows shape after sliding window: (38, 100, 9)
2025-05-09 14:42:33,303 - DEBUG - myapp - Padding codes: []
2025-05-09 14:42:33,304 - INFO - myapp - Numero di finestre estratte: 38
2025-05-09 14:42:33,307 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 0
2025-05-09 14:42:33,308 - INFO - myapp - Numero totale di finestre (dopo padding finale): 38
2025-05-09 14:42:33,311 - DEBUG - myapp - Y_windows_full shape after sliding window: (

>>> Kid_ids: [3006 3007 3008 3009 3010 3017 3022]


2025-05-09 14:42:33,470 - DEBUG - myapp - Y_windows shape after extraction: (96, 1)
2025-05-09 14:42:33,471 - INFO - myapp - Kid_id: 3009, Action_id: 10.0, Action_count: 96
2025-05-09 14:42:33,477 - INFO - myapp - Kid_id: 3010, X_kid shape: (3147, 9), Y_kid shape: (3147,)
2025-05-09 14:42:33,481 - INFO - myapp - Numero di finestre estratte: 61
2025-05-09 14:42:33,483 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 47
2025-05-09 14:42:33,487 - INFO - myapp - Padding normale applicato. Padding code: 1
2025-05-09 14:42:33,487 - DEBUG - myapp - Nuova finestra con padding: (1, 100, 9)
2025-05-09 14:42:33,487 - INFO - myapp - Numero totale di finestre (dopo padding finale): 62
2025-05-09 14:42:33,492 - DEBUG - myapp - X_windows shape after sliding window: (62, 100, 9)
2025-05-09 14:42:33,493 - DEBUG - myapp - Padding codes: [1]
2025-05-09 14:42:33,497 - INFO - myapp - Numero di finestre estratte: 61
2025-05-09 14:42:33,499 - DEBUG - myapp - Campioni rimanenti dopo l'ult

>>> Kid_ids: [3005 3008 3009 3010]


2025-05-09 14:42:33,825 - DEBUG - myapp - sto leggendo il file csv: C:\codes\HumanActivityRecognition\data\pdd_data\df_car_action_12.csv
2025-05-09 14:42:33,832 - INFO - myapp - Kid_id: 3008, X_kid shape: (535, 9), Y_kid shape: (535,)
2025-05-09 14:42:33,833 - INFO - myapp - Numero di finestre estratte: 9
2025-05-09 14:42:33,834 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 35
2025-05-09 14:42:33,837 - INFO - myapp - Padding normale applicato. Padding code: 1
2025-05-09 14:42:33,839 - DEBUG - myapp - Nuova finestra con padding: (1, 100, 9)
2025-05-09 14:42:33,840 - INFO - myapp - Numero totale di finestre (dopo padding finale): 10
2025-05-09 14:42:33,843 - DEBUG - myapp - X_windows shape after sliding window: (10, 100, 9)
2025-05-09 14:42:33,844 - DEBUG - myapp - Padding codes: [1]
2025-05-09 14:42:33,846 - INFO - myapp - Numero di finestre estratte: 9
2025-05-09 14:42:33,849 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 35
2025-05-09 14:

...

2025-05-09 14:42:36,938 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 29
2025-05-09 14:42:36,939 - INFO - myapp - Padding estremo applicato. Padding code: 2
2025-05-09 14:42:36,940 - DEBUG - myapp - Nuova finestra con padding: (1, 100, 1)
2025-05-09 14:42:36,940 - INFO - myapp - Numero totale di finestre (dopo padding finale): 48
2025-05-09 14:42:36,942 - DEBUG - myapp - Y_windows_full shape after sliding window: (48, 100, 1)
2025-05-09 14:42:36,944 - DEBUG - myapp - Padding codes: [2]
2025-05-09 14:42:36,946 - DEBUG - myapp - Y_windows shape after extraction: (48, 1)
2025-05-09 14:42:36,949 - INFO - myapp - Kid_id: 3010, Action_id: 5.0, Action_count: 48
2025-05-09 14:42:36,951 - DEBUG - myapp - Conteggio finale di finestre per ogni bambino per l'azione 5.0: {3010: 48}
2025-05-09 14:42:36,953 - INFO - myapp - Numero  totale di finestre per l'azione 5:48
2025-05-09 14:42:36,955 - INFO - myapp - Contenuto finale di kid_action_counts: {'car_action_10': {3006: 38, 3

>>> Kid_ids: [3009 3010 3017]
>>> Kid_ids: [3010 3017]


2025-05-09 14:42:37,173 - INFO - myapp - Numero di finestre estratte: 1
2025-05-09 14:42:37,175 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 0
2025-05-09 14:42:37,177 - INFO - myapp - Numero totale di finestre (dopo padding finale): 1
2025-05-09 14:42:37,178 - DEBUG - myapp - Y_windows_full shape after sliding window: (1, 100, 1)
2025-05-09 14:42:37,179 - DEBUG - myapp - Padding codes: [1]
2025-05-09 14:42:37,180 - DEBUG - myapp - Y_windows shape after extraction: (1, 1)
2025-05-09 14:42:37,188 - INFO - myapp - Kid_id: 3017, Action_id: 7.0, Action_count: 1
2025-05-09 14:42:37,190 - DEBUG - myapp - Conteggio finale di finestre per ogni bambino per l'azione 7.0: {3010: 12, 3017: 1}
2025-05-09 14:42:37,191 - INFO - myapp - Numero  totale di finestre per l'azione 7:13
2025-05-09 14:42:37,196 - INFO - myapp - Contenuto finale di kid_action_counts: {'car_action_10': {3006: 38, 3007: 26, 3008: 981, 3009: 96, 3010: 62, 3017: 44, 3022: 20}, 'car_action_11': {3005: 2, 30

>>> Kid_ids: [3008 3009 3017]
>>> Kid_ids: [3010 3017]


2025-05-09 14:42:37,423 - DEBUG - myapp - Padding codes: []
2025-05-09 14:42:37,425 - DEBUG - myapp - Y_windows shape after extraction: (36, 1)
2025-05-09 14:42:37,426 - INFO - myapp - Kid_id: 3010, Action_id: 9, Action_count: 36
2025-05-09 14:42:37,432 - INFO - myapp - Kid_id: 3017, X_kid shape: (640, 9), Y_kid shape: (640,)
2025-05-09 14:42:37,435 - INFO - myapp - Numero di finestre estratte: 11
2025-05-09 14:42:37,436 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 40
2025-05-09 14:42:37,438 - INFO - myapp - Padding normale applicato. Padding code: 1
2025-05-09 14:42:37,440 - DEBUG - myapp - Nuova finestra con padding: (1, 100, 9)
2025-05-09 14:42:37,441 - INFO - myapp - Numero totale di finestre (dopo padding finale): 12
2025-05-09 14:42:37,445 - DEBUG - myapp - X_windows shape after sliding window: (12, 100, 9)
2025-05-09 14:42:37,447 - DEBUG - myapp - Padding codes: [1]
2025-05-09 14:42:37,449 - INFO - myapp - Numero di finestre estratte: 11
2025-05-09 14:42

In [5]:
from collections import defaultdict

kid_summary = defaultdict(dict) # Inizializzo un dizionario per tenere traccia delle azioni per ogni bambino

for action, kid_counts in kid_action_counts.items():
    for kid, count in kid_counts.items():
        kid_summary[kid][action] = count

# Stampo il riepilogo per ogni bambino
for kid, actions in kid_summary.items():
    logger.info(f"Bambino {kid}:")
    for action, count in actions.items():
        logger.info(f"  {action}: {count} finestre")
    logger.info("="*50)


2025-05-09 14:42:52,818 - INFO - myapp - Bambino 3006:
2025-05-09 14:42:52,820 - INFO - myapp -   car_action_10: 38 finestre
2025-05-09 14:42:52,821 - INFO - myapp -   car_action_19: 7 finestre
2025-05-09 14:42:52,823 - INFO - myapp -   car_action_2: 236 finestre
2025-05-09 14:42:52,825 - INFO - myapp -   car_action_25: 13 finestre
2025-05-09 14:42:52,827 - INFO - myapp -   car_action_4: 15 finestre
2025-05-09 14:42:52,828 - INFO - myapp -   car_action_41: 51 finestre
2025-05-09 14:42:52,829 - INFO - myapp - ==================================================
2025-05-09 14:42:52,830 - INFO - myapp - Bambino 3007:
2025-05-09 14:42:52,832 - INFO - myapp -   car_action_10: 26 finestre
2025-05-09 14:42:52,833 - INFO - myapp -   car_action_16: 93 finestre
2025-05-09 14:42:52,834 - INFO - myapp -   car_action_41: 2 finestre
2025-05-09 14:42:52,837 - INFO - myapp - ==================================================
2025-05-09 14:42:52,838 - INFO - myapp - Bambino 3008:
2025-05-09 14:42:52,839 

*SPLIT DATASET IN TRS, VS,TS*  
*OUTPUT: train_folds (lista lunga quanto il numero di fold): Ogni elemento è una tupla (X_train, Y_train)  e così anche per val e test


In [12]:
import json
import random
from sklearn.utils import shuffle

import json
import numpy as np
from sklearn.model_selection import GroupKFold

def create_group_kfold_splits(kid_summary, output_file='groupkfold_splits.json', n_splits=5):
    """
    Creo uno split GroupKFold basato sugli ID dei bambini, 
    assicurandomi che ogni bambino compaia solo in un set per fold (train o test).
    Ogni fold ha: 70% train, 15% val, 15% test, sempre soggetto-wise.
    """
    all_kids = np.array(list(kid_summary.keys()))
    all_labels = np.zeros(len(all_kids))  # fittizio: GroupKFold ignora y
    groups = all_kids  # ogni soggetto è un gruppo

    gkf = GroupKFold(n_splits=n_splits)
    subject_splits = {}

    for i, (train_val_idx, test_idx) in enumerate(gkf.split(X=all_kids, y=all_labels, groups=groups)):
        split_name = f"split_{i}"

        train_val_kids = all_kids[train_val_idx]
        test_kids = all_kids[test_idx]

        # Ora dividiamo il train_val in 70% train e 30% val (che sarà 15% finale rispetto al totale)
        n_val = max(1, int(len(train_val_kids) * 0.3))
        val_kids = train_val_kids[:n_val]
        train_kids = train_val_kids[n_val:]

        subject_splits[split_name] = {
            'train_indices': train_kids.tolist(),
            'val_indices': val_kids.tolist(),
            'test_indices': test_kids.tolist()
        }

        logger.info(f"{split_name}: Train={len(train_kids)}, Val={len(val_kids)}, Test={len(test_kids)}")

    with open(output_file, 'w') as f:
        json.dump(subject_splits, f, indent=4)

    logger.info("Tutti gli split GroupKFold sono stati salvati.")
    return subject_splits



split = create_group_kfold_splits(kid_summary)


2025-05-09 13:00:12,897 - INFO - myapp - split_0: Train=6, Val=2, Test=2
2025-05-09 13:00:12,901 - INFO - myapp - split_1: Train=6, Val=2, Test=2
2025-05-09 13:00:12,905 - INFO - myapp - split_2: Train=6, Val=2, Test=2
2025-05-09 13:00:12,921 - INFO - myapp - split_3: Train=6, Val=2, Test=2
2025-05-09 13:00:12,923 - INFO - myapp - split_4: Train=6, Val=2, Test=2
2025-05-09 13:00:12,927 - INFO - myapp - Tutti gli split GroupKFold sono stati salvati.


In [ ]:
import json
import numpy as np

# Carico gli split salvati dal file JSON
with open('groupkfold_splits.json', 'r') as f:
    subject_splits = json.load(f)

# Inizializzo una lista per raccogliere i dati per ogni fold (train, val, test) ogni fold è una lista di tupled
train_folds = []
val_folds = []
test_folds = []

# Per ogni split (fold) in subject_splits, separo i dati in train, val, test
for split_name, split_data in subject_splits.items():
    train_kids = split_data['train_indices'] #estraggo ID dei bambini per il training
    val_kids = split_data['val_indices'] # ID dei bambini per la validazione
    test_kids = split_data['test_indices'] # ID dei bambini per il test

    # Creo le maschere booleane
    train_mask = np.isin(kid_ids, train_kids)
    val_mask = np.isin(kid_ids, val_kids)
    test_mask = np.isin(kid_ids, test_kids)

    # Applico le maschere per ottenere i dati
    X_train, Y_train = X[train_mask], Y[train_mask] #tutte le finestre per i bambini del training
    X_val, Y_val = X[val_mask], Y[val_mask]
    X_test, Y_test = X[test_mask], Y[test_mask]

    # Appiattisco le etichette se necessario
    Y_train = Y_train.flatten()  #se Y_train è multidimensionale, lo appiattisco (esempio: (100,1) diventa (100,))
    Y_val = Y_val.flatten()
    Y_test = Y_test.flatten()
 


    # Aggiungo i dati separati per questo fold alle rispettive liste
    train_folds.append((X_train, Y_train))
    val_folds.append((X_val, Y_val))
    test_folds.append((X_test, Y_test))

    logger.info(f"Fold {split_name} - Train shape: {X_train.shape}, Val shape: {X_val.shape}, Test shape: {X_test.shape}")




2025-05-09 14:54:53,939 - INFO - myapp - Fold split_0 - Train shape: (1536, 100, 9), Val shape: (481, 100, 9), Test shape: (1886, 100, 9)
2025-05-09 14:54:53,956 - INFO - myapp - Fold split_1 - Train shape: (1412, 100, 9), Val shape: (2213, 100, 9), Test shape: (278, 100, 9)
2025-05-09 14:54:53,972 - INFO - myapp - Fold split_2 - Train shape: (1386, 100, 9), Val shape: (1974, 100, 9), Test shape: (543, 100, 9)
2025-05-09 14:54:53,991 - INFO - myapp - Fold split_3 - Train shape: (2626, 100, 9), Val shape: (481, 100, 9), Test shape: (796, 100, 9)
2025-05-09 14:54:54,005 - INFO - myapp - Fold split_4 - Train shape: (3022, 100, 9), Val shape: (481, 100, 9), Test shape: (400, 100, 9)


*RIASSEGNAZIONE DELLE ETICHETTE*

In [ ]:
# TrovO tutte le etichette uniche presenti nei dati
# CreO un mapping globale delle etichette (valido per tutti i fold)
unique_labels_global = np.unique(Y)
label_mapping = {label: idx for idx, label in enumerate(unique_labels_global)}
logger.info("Mapping globale delle etichette: %s", label_mapping)

# Liste per salvare i fold con etichette remappate
train_folds_mapped = []
val_folds_mapped = []
test_folds_mapped = []

# Per ogni fold applico lo stesso mapping
for i, (train_data, val_data, test_data) in enumerate(zip(train_folds, val_folds, test_folds)):
    X_train, Y_train = train_data
    X_val, Y_val = val_data
    X_test, Y_test = test_data

    try:
        # Applico il mapping globale
        Y_train_mapped = np.array([label_mapping[y] for y in Y_train])
        Y_val_mapped = np.array([label_mapping[y] for y in Y_val])
        Y_test_mapped = np.array([label_mapping[y] for y in Y_test])
    except KeyError as e:
        logger.error(f"Errore nel fold {i}: etichetta non trovata nel mapping globale: {e}")
        raise

    # Salvo i fold remappati
    train_folds_mapped.append((X_train, Y_train_mapped))
    val_folds_mapped.append((X_val, Y_val_mapped))
    test_folds_mapped.append((X_test, Y_test_mapped))

    logger.info(f"Fold {i} - Etichette remappate: Train {np.unique(Y_train_mapped)}, Val {np.unique(Y_val_mapped)}, Test {np.unique(Y_test_mapped)}")

# TO DO: salvo mappatura 




2025-05-09 15:26:56,114 - INFO - myapp - Mapping globale delle etichette: {np.float64(2.0): 0, np.float64(3.0): 1, np.float64(4.0): 2, np.float64(5.0): 3, np.float64(6.0): 4, np.float64(7.0): 5, np.float64(8.0): 6, np.float64(9.0): 7, np.float64(10.0): 8, np.float64(11.0): 9, np.float64(12.0): 10, np.float64(13.0): 11, np.float64(14.0): 12, np.float64(16.0): 13, np.float64(18.0): 14, np.float64(19.0): 15, np.float64(21.0): 16, np.float64(23.0): 17, np.float64(25.0): 18, np.float64(27.0): 19, np.float64(28.0): 20, np.float64(29.0): 21, np.float64(31.0): 22, np.float64(32.0): 23, np.float64(37.0): 24, np.float64(38.0): 25, np.float64(39.0): 26, np.float64(40.0): 27, np.float64(41.0): 28}
2025-05-09 15:26:56,127 - INFO - myapp - Fold 0 - Etichette remappate: Train [ 0  1  2  3  4  5  6  7  8  9 11 12 14 15 16 17 18 19 20 21 22 23 25 26
 27 28], Val [ 0  2  8 13 15 18 28], Test [ 1  2  6  8  9 10 11 14 15 17 18 20 24 28]
2025-05-09 15:26:56,133 - INFO - myapp - Fold 1 - Etichette remappate

2025-05-09 15:26:56,140 - INFO - myapp - Fold 2 - Etichette remappate: Train [ 0  1  2  3  4  5  6  7  8  9 11 12 14 15 16 17 18 19 20 21 22 23 25 26
 27 28], Val [ 1  2  6  8  9 10 11 13 14 15 17 18 20 24 28], Test [ 0  1  2  8 15 17 18 23 28]
2025-05-09 15:26:56,144 - INFO - myapp - Fold 3 - Etichette remappate: Train [ 1  2  4  5  6  7  8  9 10 11 12 14 15 16 17 18 19 20 21 23 24 28], Val [ 0  2  8 13 15 18 28], Test [ 0  1  2  3  4  5  7  8  9 11 14 18 22 25 26 27 28]
2025-05-09 15:26:56,148 - INFO - myapp - Fold 4 - Etichette remappate: Train [ 0  1  2  3  4  5  6  7  8  9 10 11 12 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28], Val [ 0  2  8 13 15 18 28], Test [ 1  2  4  6  8  9 14 15 16 17 18 19 20 23]


## ***OPTUNA***

In [ ]:
BEST_MODEL_PATH = os.path.join(MODELS_DIR, "best_model_car_inference_without_norm_without_sampler_aug_4_classes.pkl")
BEST_SCORE_PATH = os.path.join(REPORTS_DIR, "best_score_car_inference_without_norm_without_sampler_aug_4_classes.txt")

best_hyperparams_file = os.path.join(REPORTS_DIR, 'best_hyperparameters_car_inference_without_norm_without_sampler_aug_4_classes.csv')
if os.path.exists(best_hyperparams_file):
    logger.debug(f"Carico i migliori iperparametri da {best_hyperparams_file}")
    best_hyperparameters=pd.read_csv(best_hyperparams_file).iloc[0].to_dict()
    best_lr = best_hyperparameters['lr']
    best_batch_size = int(best_hyperparameters['batch_size'])

else:
    def objective(trial):
        # DefiniscO gli iperparametri da ottimizzare
        lr = trial.suggest_float('lr', 1e-4, 1e-1, log=True)
        batch_size = trial.suggest_categorical('batch_size', [2,4,6])

        best_f1_score = 0

        # Crea i DataLoader per ogni fold
        for i, ((X_train, Y_train_mapped), (X_val, Y_val_mapped)) in enumerate(zip(train_folds_mapped, val_folds_mapped)):

            # Creo i dataset per questo fold
            train_dataset = HARDataset(X_train, Y_train_mapped)
            val_dataset = HARDataset(X_val, Y_val_mapped)

            # Creo i DataLoader per questo fold
            train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
            val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

            # Log delle dimensioni dei dataset
            logger.info(f"[Fold {i}] Lunghezza dataset di training: {len(train_dataset)}")
            logger.info(f"[Fold {i}] Lunghezza dataset di validazione: {len(val_dataset)}")
            # Debug
            logger.debug(f"[Fold {i}] Tipo di train_loader: {type(train_loader)}")
            logger.debug(f"[Fold {i}] Tipo di val_loader: {type(val_loader)}")


            

            # Creo il modello con gli iperparametri suggeriti e rimuovo la testa originale, in modo da non caricare i pesi associati
            model = DeepConvLSTM()

        
            model.load_state_dict(torch.load(r'C:\codes\HumanActivityRecognition\models\best_model_dl_without_norm_without_sampler_augmented_4_classes.pth', map_location=torch.device('cpu')), strict=False)



            # Ora sostituisco la testa del modello con la nuova dimensione di classi (4)
            num_ftrs = model.fc.in_features
            model.fc = nn.Linear(num_ftrs, 29)  # 4 classi
            model.set_n_classes(29)


            # Congelo tutti i parametri tranne quelli della testa (fully connected)
            for param in model.parameters():
                param.requires_grad = False  # Congelo tutti i pesi

            # Sblocco i parametri della testa (fully connected)
            for param in model.fc.parameters():
                param.requires_grad = True  # Solo i pesi della testa saranno addestrabili

            # Eseguo l'allenamento
            fold_best_f1_score = train_with_cm.train(model, train_loader, val_loader, epochs=100, batch_size=batch_size, lr=lr)



            #STAMPA CM CHE PERO' POI SI SOVRASCRIVE OGNI VOLTA CON LA MIGLIORE PER IL TRIAL MIGLIORE


            if fold_best_f1_score > best_f1_score:
                best_f1_score = fold_best_f1_score

            # Salvo modello se è il migliore finora
            is_better = False

            if not os.path.exists(BEST_SCORE_PATH) or fold_best_f1_score > float(open(BEST_SCORE_PATH).read()): #se il file non esiste o se il punteggio corrente è migliore di quello salvato 
                is_better = True
            if is_better:
                torch.save(model.state_dict(), BEST_MODEL_PATH)
                with open(BEST_SCORE_PATH, "w") as f:
                    f.write(str(fold_best_f1_score))
                logger.info(f"Nuovo miglior modello salvato in: {BEST_MODEL_PATH} con F1 score: {fold_best_f1_score}") 

                

        return best_f1_score #TODO VEDI APPUNTI (ALLA FINE HAI LA MEDIA DI TUTTI ):  VEDI OPTUNA CROSS VALIDATION

    # Creazione studio Optuna ottimizza, nel senso di minimizzare la loss in 100 prove
    # Creazione dello studio con il MedianPruner
    study = optuna.create_study(
        direction='maximize', 
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=10)  # Parametri di pruning per evitare di continuare trial non promettenti: n_startup_trials=5 significa che i primi 5 trial non verranno prunati, n_warmup_steps=10 significa che dopo 10 trial verrà applicato il pruning
        #Il pruner interromperà automaticamente i trial che non sono promettenti, basandosi sui punteggi parziali (F1-score) ottenuti durante l'allenamento.
    )
    study.optimize(objective, n_trials=100)

    logger.info("Best hyperparameters: ", study.best_params)
    logger.info("Highest F1-score: ", study.best_value)

    #salvo i best hyperparameters
    best_hyperparameters = study.best_params
    best_hyperparameters['best_f1_score'] = study.best_value
    best_hyperparameters_df = pd.DataFrame([best_hyperparameters])
    best_hyperparameters_df.to_csv(os.path.join(REPORTS_DIR, 'best_hyperparameters_inference_car_without_norm_without_sampler_aug_4_classes.csv'), index=False)

    best_lr = study.best_params['lr']
    best_batch_size = study.best_params['batch_size']

    #visualizzare la storia dell'ottimizzazione effettuata da Optuna. Ci permette di vedere come l'f1 score
    # è cambiato nel corso delle diverse prove (trials) durante l'ottimizzazione.
    file_name = "optimization_history_dl_without_norm_without_sampler_aug_4_classes.png"
    fig=vis.plot_optimization_history(study)
    plt.show()

    # Salvo il grafico nella cartella FIGURES con il nome specificato
    fig.write_image(os.path.join(FIGURES_DIR, file_name))

    logger.debug(f"Grafico salvato in figures /{file_name}")


#STAMPO CM DELL'ALLENAMENTO SUL TRAINING SET
for i, ((X_train, Y_train_mapped), (X_val, Y_val_mapped)) in enumerate(zip(train_folds_mapped, val_folds_mapped)):

    # Plot della matrice di confusione per il training set
    plot_CM(mdl_class=DeepConvLSTM, 
            mdl_weights=BEST_MODEL_PATH, 
            X=X_train, 
            Y=Y_train_mapped,
            batch_size=best_batch_size, 
            figure_name=f"cm_train_fold_{i}_best_hyp_inference_car_optuna_dl_without_norm_without_sampler_aug_4_classes")
    
    # Plot della matrice di confusione per il validation set
    plot_CM(mdl_class=DeepConvLSTM, 
            mdl_weights=BEST_MODEL_PATH, 
            X=X_val, 
            Y=Y_val_mapped,
            batch_size=best_batch_size, 
            figure_name=f"cm_val_fold_{i}_best_hyp_inference_car_optuna_dl_without_norm_without_sampler_aug_4_classes")

#

# Creo modello e carico pesi del miglior modello (trovato prima in optuna)
model = DeepConvLSTM(n_classes=4)

# Rimuovo la testa originale, in modo da non caricare i pesi associati
model.load_state_dict(
    torch.load(
        r'C:\codes\HumanActivityRecognition\models\best_model_car_inference_without_norm_without_sampler_aug_4_classes.pkl',
        map_location=torch.device('cpu')
    ),
    strict=False
)


# Congelo tutti i parametri tranne quelli della testa (fully connected)
for param in model.parameters():
        param.requires_grad = False  # Congela tutti i pesi


for name, param in model.named_parameters():
    print(f"{name} requires_grad={param.requires_grad}")



# Creo i DataLoader con i migliori iperparametri
test_loader = DataLoader(test_dataset, batch_size=best_batch_size, drop_last=True, shuffle=False)



# Definisco la funzione di perdita
criterion = nn.CrossEntropyLoss()


# Eseguo l'eval sul test set usando i migliori iperparametri
test_loss, test_acc, test_f1 = train_with_cm.evaluate_model(model, test_loader, figure_name= "cm_test_best_hyp_optuna_dl_without_norm_without_sampler_aug_4_classes", save_confusion_matrix=True)

*nella funzione obiettivo: uso train e val*  
*stampo cm di train e val*  
*train finale con dati di test con cm*


